# 03. Adjacent values

**Tier:** Deep. **Why you'd use this:** The area between two lines is the distance between them times the shorter height. Start at both ends.

**After this lesson you can:** walk adjacent pairs with two indexes; catch a local condition without nested loops.
About 20 minutes. **Uses:** [Sorted squares](02-sorted-squares-lesson.ipynb)


## Quick review

1. What does an adjacent pair mean in index terms?
2. Why is one forward pass enough for a local neighbor check?

<details><summary>Answers</summary>

For 03 adjacent lesson: skim [Sorted squares](02-sorted-squares-lesson.ipynb) if needed.
</details>


## Recognize it

You only care about neighbors that sit next to each other in index order. The prompt says "adjacent", "neighbors", "each pair of consecutive values", or a local condition on `i` and `i + 1`.

Signals: the answer is built from pairs `(nums[i], nums[i + 1])`, and a single forward pass is enough. A look-alike that is not this pattern: opposite-end pointers for a pair sum or for container area — those indexes are not required to be adjacent.

A second look-alike is "any two values", which invites a nested loop or a set. Adjacent problems name consecutive positions on purpose. If the prompt never says neighbors and distance can be larger than one, do not shrink the search to edges alone.


## Why it works this way

Adjacent pairs share an edge in the index line. There are exactly `n - 1` of them in a list of length `n`. Walking `index` from `0` to `n - 2` visits each pair once. No nested loop is required because the second index is always `index + 1`. Time stays proportional to `n`, and extra memory is constant if you only keep a running best.

The sibling question asks for the largest sum of two neighbors. At each step you compare `nums[index] + nums[index + 1]` to the best so far. Starting `best` from the first real pair avoids treating an empty or one-element list as a fake pair. Returning `0` when there is no pair is a chosen empty answer for this sibling; other problems may raise or return `None`. Seeding from `0` instead would hide an all-negative answer behind a fake non-negative floor.

Opposite-end two pointers solve a different geometry: the width between far indexes matters, and moving the shorter end is the move. Here width is always `1` in index space, so the only degree of freedom is which adjacent pair you are looking at. Sliding a fixed window of length 2 is the same algorithm under another name — the window never grows, and the update is just "drop the left neighbor, add the new right neighbor" if you tracked a sum. For a max of sums, comparing each pair directly is clearer.

Lists are array-backed, so reading `nums[index + 1]` is constant time. See [Lists](../../internals/03-lists/01-dynamic-array-lesson.ipynb) if you want the memory picture. Linked lists would need an explicit second reference; that is a different lesson.

Design-wise this is the cheapest two-pointer shape: one index, one lookahead, no inward shrink proof. The cost of overusing it is wrong answers on non-adjacent problems, not slow code. Name the adjacency requirement before you write the loop so you do not "optimize" away pairs the prompt still wants.


## When to use it

Use an adjacent pass when the condition names neighbors or consecutive positions. Use opposite-end pointers when distance between far indexes matters. Use a fixed sliding window when the window length is a parameter `k` larger than 2. Use a nested loop only when non-adjacent pairs are in scope.

If the answer needs every pair of indexes with gap at most `k`, grow a window or use two indexes that stay within that gap — a pure `i` and `i + 1` loop is too narrow. If values must be unique across the whole list, reach for a set, not neighbor sums.


## Brute force, and why it's slow

Build every index pair, then keep only those with gap 1. The extra pairs are wasted work.


In [1]:
def adjacent_max_slow(nums):
    best = None
    pairs = 0
    for i in range(len(nums)):
        for j in range(i + 1, len(nums)):
            pairs += 1
            if j == i + 1:
                total = nums[i] + nums[j]
                best = total if best is None else max(best, total)
    return (0 if best is None else best), pairs


print(adjacent_max_slow([1, 8, 7, 2]))
print(adjacent_max_slow([4]))


(15, 6)
(0, 0)


## Key insight

Every adjacent pair is `(nums[i], nums[i + 1])`. One loop over `i` visits them all; non-adjacent pairs never enter the race. Seed the best from a real pair so negatives stay eligible.


## Worked example

Return the largest sum of two neighbors, or `0` when there is no pair.


In [2]:
def adjacent_max(nums: list[int]) -> int:
    # 1. No pair means no answer under this sibling's rule.
    if len(nums) < 2:
        return 0
    # 2. Seed with the first real pair, then walk the rest.
    best = nums[0] + nums[1]
    for index in range(1, len(nums) - 1):
        best = max(best, nums[index] + nums[index + 1])
    return best


print(adjacent_max([1, 8, 7, 2]))
print(adjacent_max([4]))
print(adjacent_max([-3, -1, -2]))


15
0
-3


## Trace

`[1, 8, 7, 2]`:

| pair | sum | best |
|---|---|---|
| 1 + 8 | 9 | 9 |
| 8 + 7 | 15 | **15** |
| 7 + 2 | 9 | 15 |


## Cost

The fast pass looks at each adjacent edge once: time grows with `n`, memory stays constant. The slow double loop still walks about `n² / 2` index pairs even though only `n - 1` of them are adjacent. There is no sort and no hash table unless the sibling asks for something beyond a local best.


## Variants

Not solved here: [Container With Most Water](../../challenges/problems/05-container-with-most-water.ipynb), [Trapping Rain Water](../../challenges/problems/22-trapping-rain-water.ipynb), [3Sum](../../challenges/problems/04-3sum.ipynb).


## Common mistake

This bug is silent. The loop starts `best` at `0`, so an all-negative list wrongly reports `0` instead of the least-negative adjacent sum. There is no traceback. The print is `0`, and the answer is `-3`.


In [3]:
def adjacent_max_wrong(nums):
    if len(nums) < 2:
        return 0
    best = 0  # assumes a non-negative winner
    for index in range(len(nums) - 1):
        best = max(best, nums[index] + nums[index + 1])
    return best


print(adjacent_max_wrong([-3, -1, -2]))


0


The fix seeds `best` from the first real pair. Negatives stay in the race. The worked example does that, so `[-3, -1, -2]` prints `-3`.


In [4]:
print(adjacent_max([-3, -1, -2]))


-3


## Python idioms

`max(a, b)` updates the running best. A short window form uses a generator over adjacent pairs when you want a one-liner for the same idea.


In [5]:
def adjacent_max_gen(nums: list[int]) -> int:
    if len(nums) < 2:
        return 0
    return max(nums[i] + nums[i + 1] for i in range(len(nums) - 1))


print(adjacent_max_gen([1, 8, 7, 2]))


15


## Recap

- Adjacent means indexes that differ by one.
- One forward pass visits every neighbor pair.
- Seed the best from a real pair so negatives work.

Next: [practice](03-adjacent-practice.ipynb).
